## AY25-26 Term 2 CS421 Group Project

In [95]:
%reload_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis, entropy
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
import mlflow

from features.feature_pipeline import BaselineFeatureBuilder, InteractionFeatureBuilder
data=np.load("../input/training_batch_with_labels.npz")
print(data)

NpzFile '../input/training_batch_with_labels.npz' with keys: X, y


In [96]:
X=data["X"]
y=data["y"]

print("# of interactions:", X.shape[0])
print("# of anomalous and normal users:", np.count_nonzero(y==1), np.count_nonzero(y==0))


df=pd.DataFrame(X)
labels=pd.DataFrame(y)
df.rename(columns={0:"user",1:"item",2:"rating"},inplace=True)
print("# of items:", df['item'].unique().shape[0])

# of interactions: 177346
# of anomalous and normal users: 100 1000
# of items: 993


In [97]:
df.head(100)
labels

,0,1
0,100,0
1,101,0
2,102,0
3,103,0
4,104,0
...,...,...
1095,1195,0
1096,1196,0
1097,1197,0
1098,1198,0


In [98]:
labels.rename(columns={0:"user",1:"label"},inplace=True)

In [103]:
labels

,user,label
0,100,0
1,101,0
2,102,0
3,103,0
4,104,0
...,...,...
1095,1195,0
1096,1196,0
1097,1197,0
1098,1198,0


## Feature Engineering

The objective is to derive user-level behavioural features from the interaction dataset (user, item, rating) so that the model can learn behavioural patterns that distinguish anomalous users from normal users.

Since anomaly detection occurs at the user level, all features must summarize a user's rating behaviour across items.

Derive BaseLine user level features.

In [102]:
bf = BaselineFeatureBuilder()
user_features_baseline = bf.transform(df)
user_features_baseline.head()


/Users/Kathir/Documents/Coursework/machine_learning/project/src/features/feature_pipeline.py:29: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  rating_5_pct=("rating", lambda x: (x == 5).mean()),
/Users/Kathir/Documents/Coursework/machine_learning/project/src/features/feature_pipeline.py:30: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  


,user,num_unique_items_rated,mean_rating,median_rating,std_rating,min_rating,max_rating,high_rating_ratio,low_rating_ratio,rating_0_pct,rating_1_pct,rating_4_pct,rating_5_pct,user_rating_skew,user_rating_kurt,rating_entropy,interaction_density
0,100,88,3.147727,3.0,1.169928,0.0,5.0,0.431818,0.204545,0.045455,0.056818,0.363636,0.068182,-0.938904,0.737335,1.455461,0.088
1,101,231,3.406926,3.0,0.854016,0.0,5.0,0.432900,0.077922,0.008658,0.017316,0.346320,0.086580,-0.548093,1.864121,1.193838,0.231
2,102,98,3.673469,4.0,1.072385,1.0,5.0,0.683673,0.142857,0.000000,0.061224,0.489796,0.193878,-0.935300,0.403276,1.347083,0.098
3,103,103,3.000000,3.0,1.290994,0.0,5.0,0.417476,0.271845,0.058252,0.097087,0.349515,0.067961,-0.741755,-0.127405,1.555827,0.103
4,104,89,3.719101,4.0,0.988442,1.0,5.0,0.584270,0.056180,0.000000,0.044944,0.348315,0.235955,-0.622146,0.523690,1.265745,0.089


Now that we have our user behavioural features, we can start builing a baseline model.


In [106]:
# user_features.dtypes
# #  Separate Features and labels
y = labels['label']
X = user_features_baseline.drop(columns=["user"])
print(X.columns, len(X.columns))
X.isnull().sum()

# Since dataset is imbalanced we startify based on y
# to ensure both train and test set have the same anomaly ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
# Perform standardization of features
scaler = StandardScaler()

# fit -> learn parameters(mean, std) from data
# transform → apply transformation using those parameters
X_train_scaled = scaler.fit_transform(X_train)
# transform only no fitting occurs so that data leakage does not happen, 
# test data uses params of training data and tranforms
X_test_scaled = scaler.transform(X_test)



# Balanced weighting increases penalty for misclassifying anomalies.
model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000
)

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:,1]
print(y_prob[:5])

Index(['num_unique_items_rated', 'mean_rating', 'median_rating', 'std_rating',
       'min_rating', 'max_rating', 'high_rating_ratio', 'low_rating_ratio',
       'rating_0_pct', 'rating_1_pct', 'rating_4_pct', 'rating_5_pct',
       'user_rating_skew', 'user_rating_kurt', 'rating_entropy',
       'interaction_density'],
      dtype='object') 16
[0.08925675 0.94584407 0.93527457 0.20072961 0.03731888]


In [107]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual Normal", "Actual Anomaly"],
    columns=["Predicted Normal", "Predicted Anomaly"]
)
print("Confusion Matrix:")
print(cm_df)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ROC-AUC
roc_auc = roc_auc_score(y_test, y_prob)
print("\nROC-AUC:", roc_auc)

# AUPRC (Average Precision Score)
auprc = average_precision_score(y_test, y_prob)
print("AUPRC:", auprc)

Confusion Matrix:
                Predicted Normal  Predicted Anomaly
Actual Normal                167                 33
Actual Anomaly                 8                 12

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.83      0.89       200
           1       0.27      0.60      0.37        20

    accuracy                           0.81       220
   macro avg       0.61      0.72      0.63       220
weighted avg       0.89      0.81      0.84       220


ROC-AUC: 0.80325
AUPRC: 0.5727802654095936


## Experimenting with baseline features using XGBoost

## Train XGBoost with Item and Interaction level features

In [109]:
users = user_features_baseline["user"]
y = labels['label']

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
roc_scores = []
pr_scores = []
feature_importance_list = []
feature_names = None


model_params= {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.01,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "scale_pos_weight": 17,
    "random_state": 42
}

for fold, (train_idx, val_idx) in enumerate(kf.split(users, y)):

    print(f"\nFold {fold+1}")

    train_users = users.iloc[train_idx]
    val_users = users.iloc[val_idx]

    # ---------- SPLIT RAW DATA ----------
    train_df = df[df["user"].isin(train_users)].copy()
    val_df   = df[df["user"].isin(val_users)].copy()

    
    ibf = InteractionFeatureBuilder()
    ibf.fit(train_df)
    interaction_features_train = ibf.transform(train_df)
    interaction_features_val = ibf.transform(val_df)

    # merge with baseline features
    train_features = user_features_baseline[
        user_features_baseline["user"].isin(train_users)
    ].merge(interaction_features_train, on="user", how="left")

    val_features = user_features_baseline[
        user_features_baseline["user"].isin(val_users)
    ].merge(interaction_features_val, on="user", how="left")

    X_train = train_features.drop(columns=["user"])
    y_train = y[train_idx]

    if feature_names is None:
        feature_names = X_train.columns

    X_val = val_features.drop(columns=["user"])
    y_val = y[val_idx]


    model = xgb.XGBClassifier(**model_params)

    model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_val)[:, 1]

    roc = roc_auc_score(y_val, y_prob)
    pr = average_precision_score(y_val, y_prob)

    roc_scores.append(roc)
    pr_scores.append(pr)
    feature_importance_list.append(model.feature_importances_)

    print(f"ROC-AUC: {roc:.4f}, PR-AUC: {pr:.4f}")


print("\nFinal Results:")
print("Mean ROC-AUC:", sum(roc_scores)/len(roc_scores))
print("Mean PR-AUC:", sum(pr_scores)/len(pr_scores))

mean_importance = np.mean(feature_importance_list, axis=0)

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": mean_importance
}).sort_values(by="importance", ascending=False)
feature_importance_df



Fold 1
ROC-AUC: 0.8808, PR-AUC: 0.6756

Fold 2
ROC-AUC: 0.9220, PR-AUC: 0.6709

Fold 3
ROC-AUC: 0.9238, PR-AUC: 0.7823

Fold 4
ROC-AUC: 0.9090, PR-AUC: 0.6863

Fold 5
ROC-AUC: 0.8670, PR-AUC: 0.6504

Final Results:
Mean ROC-AUC: 0.9005000000000001
Mean PR-AUC: 0.6931115882426107


,feature,importance
9,rating_1_pct,0.101745
4,min_rating,0.068678
13,user_rating_kurt,0.050981
19,std_normalized_deviation,0.050056
24,negative_dev_ratio,0.049624
14,rating_entropy,0.043459
20,mean_abs_deviation,0.042208
7,low_rating_ratio,0.042189
8,rating_0_pct,0.041487
23,p90_abs_deviation,0.041412


# Feature Importance Interpretation (Tier 1)

## Overview

The top features can be divided into two fundamental signal types:

- **Direct anomaly indicators (low-rating behavior)**
- **Behavioral structure & inconsistency signals**

# Tier 1A — Direct Anomaly Signals

## 1. `min_rating`

**Definition:**  
Minimum rating given by the user.

**Interpretation:**  
- Captures whether the user has ever given an extreme low rating.
- Even a *single* very low rating is highly predictive of anomaly.

**Behavioral Insight:**  
- Anomalous users tend to produce **extreme negative events**.
- This is a **high-signal, low-noise feature**.


## 2. `rating_1_pct`

**Definition:**  
Fraction of ratings equal to 1.

**Interpretation:**  
- Measures how frequently the user gives low ratings.
- Stronger than `rating_0_pct`, indicating “1” is more informative.

**Behavioral Insight:**  
- Anomalies are **consistently negative**, not just occasionally.
- Frequency of low ratings is more important than average rating.

# Tier 1B — Behavioral Structure Signals

## 3. `std_normalized_deviation`

**Definition:**  
Standard deviation of normalized deviations from item mean ratings.

**Interpretation:**  
- Measures how inconsistently a user deviates from consensus.
- High value → user behaves unpredictably across items.

**Behavioral Insight:**  
- Anomalous users are **inconsistent relative to population behavior**.
- Variability matters more than average deviation.

## 4. `user_rating_kurt`

**Definition:**  
Kurtosis of the user’s rating distribution.

**Interpretation:**  
- High kurtosis → ratings concentrated at extremes (e.g., only 1s and 5s).

**Behavioral Insight:**  
- Anomalies exhibit **extreme or bimodal behavior**.
- Indicates non-natural rating patterns.

## 5. `rating_entropy`

**Definition:**  
Entropy of the rating distribution.

**Interpretation:**  
- Low entropy → repetitive behavior (e.g., always same rating)
- High entropy → random/unstructured behavior

**Behavioral Insight:**  
- Anomalies may be:
  - overly consistent (scripted behavior)
  - overly random (no pattern)

## 6. `rating_0_pct`

**Definition:**  
Fraction of ratings equal to 0.

**Interpretation:**  
- Another indicator of extreme low-rating behavior.
- Less important than `rating_1_pct`.

**Behavioral Insight:**  
- Confirms importance of **negative rating patterns**
- Suggests “0” ratings are either rarer or noisier than “1”


# Key Insights

## 1. Dominant Signal

```text
Anomalies are primarily characterized by low-rating behavior

## Training on entire dataset

In [112]:
# Use full data
train_df = df.copy()
# compute item stats again
ibf_full_data = InteractionFeatureBuilder()
ibf_full_data.fit(train_df)
interaction_features = ibf_full_data.transform(train_df)

# merge with user_level_features_baseline
full_features = user_features_baseline.merge(
    interaction_features,
    on="user",
    how="left"
).fillna(0)

X = full_features.drop(columns=["user"])
y = labels['label']

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=10,
    random_state=42
)

model.fit(X, y)

#  Load test df
test_data = np.load('../input/first_batch.npz')
test_data = test_data['X']
test_df = pd.DataFrame(
    test_data,
    columns=["user","item","rating"]
)

test_bf = BaselineFeatureBuilder()
user_features_baseline_test = test_bf.transform(test_df)
user_features_baseline_test

# Apply SAME feature logic using TRAIN statistics
test_interaction_features = ibf_full_data.transform(test_df)
test_features = user_features_baseline_test.merge(
    test_interaction_features,
    on="user",
    how="left"
).fillna(0)


batch1_test = test_features.drop(columns=["user"])

batch1_test_probs = model.predict_proba(batch1_test)[:, 1]
print(batch1_test_probs[:5])
np.savez('submission.npz', predictions=batch1_test_probs)
test_features.shape

/Users/Kathir/Documents/Coursework/machine_learning/project/src/features/feature_pipeline.py:29: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  rating_5_pct=("rating", lambda x: (x == 5).mean()),
/Users/Kathir/Documents/Coursework/machine_learning/project/src/features/feature_pipeline.py:30: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  


[0.01622496 0.19809645 0.00164745 0.00201834 0.66349214]


(1100, 27)

# Experiment 1 (Baseline + Interaction Features)

## Objective
Establish a strong baseline using:
- User-level behavioral features  
- Interaction-level deviation features  

This serves as the reference before adding pattern-based features.


## Dataset
- ~1100 users (1000 normal, 100 anomalous)  
- ~1000 items  
- Input: `(user, item, rating ∈ [0–5])`

## Features Used

### 1. User-Level Features
**Activity**
- `num_unique_items_rated`, `interaction_density`

**Statistics**
- `mean_rating`, `median_rating`, `std_rating`, `min_rating`, `max_rating`

**Distribution**
- `rating_0_pct` → `rating_5_pct`
- `high_rating_ratio`, `low_rating_ratio`

**Shape**
- `rating_entropy`, `user_rating_kurt`, `user_rating_skew`


### 2. Interaction-Level Features

**Item stats (train-only per fold)**
- `item_mean_rating`, `item_std_rating`, `item_popularity`

**Interaction signals**
- `rating_deviation = rating − item_mean`
- `normalized_deviation = rating_deviation / item_std`
- `abs_deviation = |rating_deviation|`

**User aggregation**
- `mean_rating_deviation`, `std_rating_deviation`
- `mean_normalized_deviation`, `std_normalized_deviation`
- `mean_abs_deviation`, `avg_item_popularity`


## Model
- XGBoost  
- Stratified K-Fold (user-level split)  
- Leakage-safe feature computation  


## Performance
- **ROC-AUC ≈ 0.883**  
- **PR-AUC ≈ 0.686**


## Key Learnings

### 1. Strongest Signals
- `min_rating`, `rating_1_pct`, `rating_0_pct`  
→ Detect **low-rating anomalies**

### 2. Distribution Signals
- `entropy`, `kurtosis`, `skew`  
→ Capture **behavioral patterns**


### 3. Interaction Signals
- `std_normalized_deviation`, `mean_abs_deviation`  
→ Capture **inconsistency vs item norms**

## Limitation

Current features capture:
- ✔ average behavior  
- ✔ variability  

Current features capture:
- ✔ average behavior  
- ✔ variability of behavior  

But they do **not explicitly capture event frequency**:

```text
❌ how often a user exhibits extreme anomalous behavior



Motivation:
Add features that explicitly count how often extreme deviations occur,
because anomalies are often driven by these rare events.